# Traffic Simulation Notebook

This notebook simulates various traffic patterns to test router performance under load:

1. Setup and configuration
2. Constant rate traffic simulation
3. Ramp-up traffic pattern
4. Spike traffic pattern  
5. Wave traffic pattern
6. Analysis and visualization

**Use cases:**
- Performance testing
- Latency characterization
- Throughput analysis
- Model selection distribution

**Sources:**
- `synthetic`: Random generated samples
- `db`: Real samples from database

In [ ]:
# Imports
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from artemis_router.config import load_config
from artemis_router.router_engine import RouterEngine
from artemis_router.traffic_simulator import (
    run_traffic,
    run_traffic_pattern,
    make_synthetic_sample,
)
from artemis_router.db_io import load_samples_from_db

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from collections import defaultdict

print("Imports successful!")

## 1. Configuration and Setup

In [ ]:
# Load configuration
config_path = "../router_config_example.yaml"
cfg = load_config(config_path)

print(f"Configuration loaded")
print(f"Device: {cfg.router.device}")
print(f"Models: {cfg.router.model_name_order}")

# Initialize engine
engine = RouterEngine(cfg)
print("\nRouter engine initialized!")

## 2. Simulation Parameters

Configure the traffic simulation parameters here:

In [ ]:
# Simulation parameters (adjust these)
SOURCE = "synthetic"  # "synthetic" or "db"
RPS = 10.0            # Requests per second
DURATION = 30         # Simulation duration in seconds
SPLIT = "test"        # For DB source only
DB_LIMIT = 1000       # Max samples to load from DB

print(f"Simulation parameters:")
print(f"  Source: {SOURCE}")
print(f"  Target RPS: {RPS}")
print(f"  Duration: {DURATION}s")
if SOURCE == "db":
    print(f"  DB split: {SPLIT}")
    print(f"  DB limit: {DB_LIMIT}")

In [ ]:
# Load DB samples if needed
db_samples = None
if SOURCE == "db":
    print(f"Loading samples from database...")
    db_samples = load_samples_from_db(
        engine.db_engine,
        cfg.data,
        split=SPLIT,
        limit=DB_LIMIT
    )
    print(f"Loaded {len(db_samples)} samples from DB")
else:
    print("Using synthetic samples (generated on-the-fly)")

## 3. Constant Rate Traffic

In [ ]:
print(f"Running constant traffic simulation: {RPS} RPS for {DURATION}s")
print("="*60)

results, stats = run_traffic(
    route_fn=engine.route_sample,
    source=SOURCE,
    samples=db_samples,
    traffic_cfg=cfg.traffic,
    rps=RPS,
    duration_sec=DURATION,
    verbose=True,
)

print("\n" + "="*60)
print("Simulation complete!")
print("="*60)

In [ ]:
# Detailed statistics
print(f"\nDetailed Statistics:")
print(f"  Total samples processed: {stats.total_samples}")
print(f"  Total duration: {stats.total_duration_sec:.2f}s")
print(f"  Actual RPS: {stats.actual_rps:.2f}")
print(f"  Target RPS: {RPS}")
print(f"  RPS accuracy: {(stats.actual_rps / RPS) * 100:.1f}%")
print(f"\nLatency:")
print(f"  Mean: {stats.avg_latency_ms:.2f} ms")
print(f"  P50:  {stats.p50_latency_ms:.2f} ms")
print(f"  P95:  {stats.p95_latency_ms:.2f} ms")
print(f"  P99:  {stats.p99_latency_ms:.2f} ms")
print(f"\nErrors: {stats.errors}")
print(f"\nModel distribution:")
for model, count in sorted(stats.model_distribution.items(), key=lambda x: x[1], reverse=True):
    pct = (count / stats.total_samples) * 100
    print(f"  {model:30s}: {count:4d} ({pct:5.1f}%)")

In [ ]:
# Visualize results
latencies = [r.router_decision.inference_ms for r in results]
timestamps = [i for i in range(len(results))]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Latency over time
axes[0, 0].plot(timestamps, latencies, alpha=0.6)
axes[0, 0].axhline(y=stats.avg_latency_ms, color='r', linestyle='--', label=f'Mean: {stats.avg_latency_ms:.2f}ms')
axes[0, 0].axhline(y=stats.p95_latency_ms, color='orange', linestyle='--', label=f'P95: {stats.p95_latency_ms:.2f}ms')
axes[0, 0].set_xlabel('Request #')
axes[0, 0].set_ylabel('Latency (ms)')
axes[0, 0].set_title('Router Latency Over Time')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Latency histogram
axes[0, 1].hist(latencies, bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(x=stats.avg_latency_ms, color='r', linestyle='--', label=f'Mean')
axes[0, 1].axvline(x=stats.p95_latency_ms, color='orange', linestyle='--', label=f'P95')
axes[0, 1].set_xlabel('Latency (ms)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Latency Distribution')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. Model distribution
models = list(stats.model_distribution.keys())
counts = list(stats.model_distribution.values())
axes[1, 0].bar(models, counts)
axes[1, 0].set_xlabel('Model')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Model Selection Distribution')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(alpha=0.3, axis='y')

# 4. Cumulative samples over time
cumulative = list(range(1, len(results) + 1))
time_elapsed = [i / stats.actual_rps for i in cumulative]
axes[1, 1].plot(time_elapsed, cumulative)
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('Cumulative Samples')
axes[1, 1].set_title(f'Throughput ({stats.actual_rps:.2f} RPS)')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Ramp-Up Traffic Pattern

Gradually increase traffic from base RPS to 4x base RPS.

In [ ]:
# Ramp-up simulation
BASE_RPS = 5.0
RAMP_DURATION = 60  # Total duration

print(f"Running ramp-up traffic pattern")
print(f"Base RPS: {BASE_RPS}, Duration: {RAMP_DURATION}s")
print("="*60)

ramp_phases = run_traffic_pattern(
    route_fn=engine.route_sample,
    source=SOURCE,
    samples=db_samples,
    traffic_cfg=cfg.traffic,
    pattern="ramp",
    base_rps=BASE_RPS,
    duration_sec=RAMP_DURATION,
    verbose=True,
)

print("\nRamp-up complete!")

In [ ]:
# Analyze ramp-up phases
print("\nRamp-up phase analysis:")
print("="*60)

phase_data = []
for i, (phase_results, phase_stats) in enumerate(ramp_phases):
    multiplier = i + 1
    target_rps = BASE_RPS * multiplier
    
    phase_data.append({
        'Phase': i + 1,
        'Target RPS': target_rps,
        'Actual RPS': phase_stats.actual_rps,
        'Samples': phase_stats.total_samples,
        'Avg Latency (ms)': phase_stats.avg_latency_ms,
        'P95 Latency (ms)': phase_stats.p95_latency_ms,
        'P99 Latency (ms)': phase_stats.p99_latency_ms,
        'Errors': phase_stats.errors,
    })
    
    print(f"\nPhase {i+1} (Target: {target_rps:.1f} RPS):")
    print(f"  Actual RPS: {phase_stats.actual_rps:.2f}")
    print(f"  Avg latency: {phase_stats.avg_latency_ms:.2f} ms")
    print(f"  P95 latency: {phase_stats.p95_latency_ms:.2f} ms")
    print(f"  Errors: {phase_stats.errors}")

phase_df = pd.DataFrame(phase_data)
print("\nPhase Summary Table:")
display(phase_df)

In [ ]:
# Visualize ramp-up
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Latency vs RPS
axes[0].plot(phase_df['Actual RPS'], phase_df['Avg Latency (ms)'], 'o-', label='Avg', markersize=8)
axes[0].plot(phase_df['Actual RPS'], phase_df['P95 Latency (ms)'], 's-', label='P95', markersize=8)
axes[0].plot(phase_df['Actual RPS'], phase_df['P99 Latency (ms)'], '^-', label='P99', markersize=8)
axes[0].set_xlabel('Actual RPS')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Latency vs Load (Ramp-up)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Throughput accuracy
throughput_accuracy = (phase_df['Actual RPS'] / phase_df['Target RPS']) * 100
axes[1].bar(phase_df['Phase'], throughput_accuracy)
axes[1].axhline(y=100, color='r', linestyle='--', label='Target')
axes[1].set_xlabel('Phase')
axes[1].set_ylabel('Throughput Accuracy (%)')
axes[1].set_title('Achieved vs Target RPS')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Spike Traffic Pattern

Simulate a sudden spike: normal → 10x → normal

In [ ]:
# Spike simulation
SPIKE_BASE_RPS = 5.0
SPIKE_DURATION = 60

print(f"Running spike traffic pattern")
print(f"Base RPS: {SPIKE_BASE_RPS}, Spike: {SPIKE_BASE_RPS * 10}, Duration: {SPIKE_DURATION}s")
print("="*60)

spike_phases = run_traffic_pattern(
    route_fn=engine.route_sample,
    source=SOURCE,
    samples=db_samples,
    traffic_cfg=cfg.traffic,
    pattern="spike",
    base_rps=SPIKE_BASE_RPS,
    duration_sec=SPIKE_DURATION,
    verbose=True,
)

print("\nSpike pattern complete!")

In [ ]:
# Analyze spike pattern
print("\nSpike pattern analysis:")
print("="*60)

spike_labels = ["Normal", "Spike (10x)", "Normal"]

for i, ((phase_results, phase_stats), label) in enumerate(zip(spike_phases, spike_labels)):
    print(f"\n{label}:")
    print(f"  Actual RPS: {phase_stats.actual_rps:.2f}")
    print(f"  Avg latency: {phase_stats.avg_latency_ms:.2f} ms")
    print(f"  P95 latency: {phase_stats.p95_latency_ms:.2f} ms")
    print(f"  P99 latency: {phase_stats.p99_latency_ms:.2f} ms")
    print(f"  Errors: {phase_stats.errors}")

# Combine all latencies for spike visualization
all_spike_latencies = []
all_spike_labels = []
for i, (phase_results, _) in enumerate(spike_phases):
    latencies = [r.router_decision.inference_ms for r in phase_results]
    all_spike_latencies.extend(latencies)
    all_spike_labels.extend([spike_labels[i]] * len(latencies))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Latency over time
axes[0].plot(all_spike_latencies, alpha=0.6)
# Add vertical lines for phase boundaries
phase1_end = len(spike_phases[0][0])
phase2_end = phase1_end + len(spike_phases[1][0])
axes[0].axvline(x=phase1_end, color='r', linestyle='--', label='Phase boundary')
axes[0].axvline(x=phase2_end, color='r', linestyle='--')
axes[0].set_xlabel('Request #')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Latency During Spike Pattern')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot comparison
spike_data = []
for i, (phase_results, _) in enumerate(spike_phases):
    latencies = [r.router_decision.inference_ms for r in phase_results]
    spike_data.append(latencies)

axes[1].boxplot(spike_data, labels=spike_labels)
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Latency Distribution by Phase')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Wave Traffic Pattern

Oscillating traffic: 1x → 1.5x → 2x → 1.5x → 1x → 0.5x

In [ ]:
# Wave simulation
WAVE_BASE_RPS = 10.0
WAVE_DURATION = 120

print(f"Running wave traffic pattern")
print(f"Base RPS: {WAVE_BASE_RPS}, Duration: {WAVE_DURATION}s")
print("="*60)

wave_phases = run_traffic_pattern(
    route_fn=engine.route_sample,
    source=SOURCE,
    samples=db_samples,
    traffic_cfg=cfg.traffic,
    pattern="wave",
    base_rps=WAVE_BASE_RPS,
    duration_sec=WAVE_DURATION,
    verbose=True,
)

print("\nWave pattern complete!")

In [ ]:
# Visualize wave pattern
wave_multipliers = [1.0, 1.5, 2.0, 1.5, 1.0, 0.5]

# Collect all latencies
all_wave_latencies = []
phase_boundaries = [0]
for phase_results, _ in wave_phases:
    latencies = [r.router_decision.inference_ms for r in phase_results]
    all_wave_latencies.extend(latencies)
    phase_boundaries.append(len(all_wave_latencies))

# Plot
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Latency over time
axes[0].plot(all_wave_latencies, alpha=0.6, linewidth=0.5)
# Add phase boundaries
for boundary in phase_boundaries[1:-1]:
    axes[0].axvline(x=boundary, color='r', linestyle='--', alpha=0.3)
axes[0].set_xlabel('Request #')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Latency During Wave Pattern')
axes[0].grid(alpha=0.3)

# Phase statistics
phase_stats_list = [stats for _, stats in wave_phases]
actual_rps = [s.actual_rps for s in phase_stats_list]
avg_latencies = [s.avg_latency_ms for s in phase_stats_list]
p95_latencies = [s.p95_latency_ms for s in phase_stats_list]

x = range(len(wave_phases))
axes[1].plot(x, actual_rps, 'o-', label='Actual RPS', markersize=8)
ax2 = axes[1].twinx()
ax2.plot(x, avg_latencies, 's-', color='orange', label='Avg Latency', markersize=8)
ax2.plot(x, p95_latencies, '^-', color='red', label='P95 Latency', markersize=8)

axes[1].set_xlabel('Phase')
axes[1].set_ylabel('RPS', color='blue')
ax2.set_ylabel('Latency (ms)', color='orange')
axes[1].set_title('RPS and Latency Across Wave Phases')
axes[1].grid(alpha=0.3)
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 7. Summary and Recommendations

In [ ]:
print("="*60)
print("TRAFFIC SIMULATION SUMMARY")
print("="*60)

print(f"\nConfiguration:")
print(f"  Device: {cfg.router.device}")
print(f"  Dtype: {cfg.router.dtype}")
print(f"  Models: {len(cfg.router.model_name_order)}")
print(f"  Source: {SOURCE}")

print(f"\nConstant Rate Performance ({RPS} RPS):")
print(f"  Actual RPS: {stats.actual_rps:.2f}")
print(f"  Avg latency: {stats.avg_latency_ms:.2f} ms")
print(f"  P95 latency: {stats.p95_latency_ms:.2f} ms")
print(f"  P99 latency: {stats.p99_latency_ms:.2f} ms")

if ramp_phases:
    print(f"\nRamp-Up Pattern:")
    max_phase = ramp_phases[-1][1]
    print(f"  Max RPS achieved: {max_phase.actual_rps:.2f}")
    print(f"  Max phase P95: {max_phase.p95_latency_ms:.2f} ms")

if spike_phases:
    print(f"\nSpike Pattern:")
    spike_phase = spike_phases[1][1]
    print(f"  Spike RPS: {spike_phase.actual_rps:.2f}")
    print(f"  Spike P95: {spike_phase.p95_latency_ms:.2f} ms")
    print(f"  Spike P99: {spike_phase.p99_latency_ms:.2f} ms")

print(f"\nRecommendations:")
print(f"  1. Router can handle sustained load of ~{stats.actual_rps:.0f} RPS")
if stats.p99_latency_ms < 100:
    print(f"  2. ✓ P99 latency is good (<100ms)")
else:
    print(f"  2. ⚠ P99 latency is high (>100ms) - consider optimization")

if stats.errors == 0:
    print(f"  3. ✓ No errors encountered")
else:
    print(f"  3. ⚠ {stats.errors} errors - investigate")

print(f"\n" + "="*60)
print("Traffic simulation complete!")
print("="*60)